In [2]:
import torch

In [41]:
words = open("names.txt", 'r').read().splitlines()
chars = sorted(list(set(''.join(words)))) 
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [42]:
input, target = [], []
for w in words[:1]:
  chs = ['.'] + list(w) + ['.'] 
  for ch1, ch2 in zip(chs, chs[1:]):
      ix1 = stoi[ch1]
      ix2 = stoi[ch2]
      print(ch1, ch2)
      input.append(ix1)
      target.append(ix2)

input = torch.tensor(input)
target = torch.tensor(target)

. e
e m
m m
m a
a .


In [43]:
input

tensor([ 0,  5, 13, 13,  1])

In [44]:
target

tensor([ 5, 13, 13,  1,  0])

In [45]:
import torch.nn.functional as F 
xenc = F.one_hot(input, num_classes=27).float() #Using the inbuilt pyTorch functional class we use one hot encoding to input the int into the NN
xenc

tensor([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0.]])

In [46]:
xenc.shape

torch.Size([5, 27])

In [47]:
xenc.dtype

torch.float32

In [48]:
# Passing the input through one single neuron
W = torch.randn((27, 1)) #Random weights for the NN, randn to produce random numbers in a normal distribution
xenc @ W # @ is used to multiply matrices in torch

tensor([[-0.9250],
        [-0.7542],
        [ 1.3208],
        [ 1.3208],
        [ 1.1128]])

In [49]:
#Passing the inputs through a layer of 27 neurons
W = torch.randn((27,27)) 
xenc @ W

tensor([[-1.1907, -1.4751, -0.7009,  1.1906, -0.8907,  0.6717,  0.3615,  0.5960,
         -1.2088, -0.9538,  0.8071, -0.4492, -1.3736, -0.3818,  1.5727, -0.0638,
          1.5256, -0.3949, -0.1113,  0.6949, -2.0239, -0.1661, -1.2753,  0.5644,
         -0.5395, -1.2721, -1.4871],
        [ 0.2230, -0.2562,  0.5463,  1.1602, -1.2800,  1.3771, -1.2425, -0.3108,
          0.9061,  0.4555, -1.3726, -0.2068, -0.1566, -0.7476, -1.1310,  0.4329,
          0.6014,  0.4971, -0.6061, -1.7974,  0.2713,  0.4674, -0.9895, -0.9120,
         -0.6332, -0.3526, -0.2557],
        [ 0.0897,  0.7278, -1.1914, -1.3850,  0.6517,  0.3720, -1.5461,  1.0666,
         -0.5054,  0.3470,  1.8535,  1.0201,  0.3499, -0.3695,  0.3389, -0.2449,
         -1.0028,  0.4957,  0.3684,  1.1748, -0.5004,  1.5043, -1.2947,  0.2034,
          0.8483, -0.7866,  0.7897],
        [ 0.0897,  0.7278, -1.1914, -1.3850,  0.6517,  0.3720, -1.5461,  1.0666,
         -0.5054,  0.3470,  1.8535,  1.0201,  0.3499, -0.3695,  0.3389, -0.2449

In [50]:
# Instead of having counts of the bigrams like in the 1st method, which is not possible here since the weights are randomly decided which are also negative,
# We also cannot have a probably dist of the outputs since they cannot be normalized and have sum = 1
# So we use the method of taking the log of counts and then increasing them exponentially 
# This makes it so that, all the negative numbers become <1 and posive numbers become >1

In [51]:
logits = xenc @ W # log-counts
counts = logits.exp() # equivalent N
probs = counts / counts.sum(1, keepdims=True)
probs

tensor([[0.0094, 0.0070, 0.0153, 0.1013, 0.0126, 0.0603, 0.0442, 0.0559, 0.0092,
         0.0119, 0.0690, 0.0197, 0.0078, 0.0210, 0.1484, 0.0289, 0.1416, 0.0208,
         0.0276, 0.0617, 0.0041, 0.0261, 0.0086, 0.0542, 0.0180, 0.0086, 0.0070],
        [0.0411, 0.0255, 0.0568, 0.1050, 0.0092, 0.1304, 0.0095, 0.0241, 0.0814,
         0.0519, 0.0083, 0.0268, 0.0281, 0.0156, 0.0106, 0.0507, 0.0601, 0.0541,
         0.0180, 0.0055, 0.0432, 0.0525, 0.0122, 0.0132, 0.0175, 0.0231, 0.0255],
        [0.0249, 0.0471, 0.0069, 0.0057, 0.0436, 0.0330, 0.0048, 0.0661, 0.0137,
         0.0322, 0.1451, 0.0631, 0.0323, 0.0157, 0.0319, 0.0178, 0.0083, 0.0373,
         0.0329, 0.0736, 0.0138, 0.1024, 0.0062, 0.0279, 0.0531, 0.0104, 0.0501],
        [0.0249, 0.0471, 0.0069, 0.0057, 0.0436, 0.0330, 0.0048, 0.0661, 0.0137,
         0.0322, 0.1451, 0.0631, 0.0323, 0.0157, 0.0319, 0.0178, 0.0083, 0.0373,
         0.0329, 0.0736, 0.0138, 0.1024, 0.0062, 0.0279, 0.0531, 0.0104, 0.0501],
        [0.0193, 0.0145,

In [52]:
probs.shape

torch.Size([5, 27])

In [53]:
# Getting the probablities of the next character, for every single input we give

In [54]:
# Ultimate process taking place:
# An input is achieved, sent through one hot encoding, entered into the NN, a list of prob. are achieved based on weight.input and the NN decides the next char

In [55]:
nlls = torch.zeros(5)
for i in range(5):
  # i-th bigram:
  x = input[i].item() # input character index
  y = target[i].item() # label character index
  print('--------')
  print(f'bigram example {i+1}: {itos[x]}{itos[y]} (indexes {x},{y})')
  print('input to the neural net:', x)
  print('output probabilities from the neural net:', probs[i])
  print('label (actual next character):', y)
  p = probs[i, y]
  print('probability assigned by the net to the the correct character:', p.item())
  logp = torch.log(p)
  print('log likelihood:', logp.item())
  nll = -logp
  print('negative log likelihood:', nll.item())
  nlls[i] = nll

print('=========')
print('average negative log likelihood, i.e. loss =', nlls.mean().item())

--------
bigram example 1: .e (indexes 0,5)
input to the neural net: 0
output probabilities from the neural net: tensor([0.0094, 0.0070, 0.0153, 0.1013, 0.0126, 0.0603, 0.0442, 0.0559, 0.0092,
        0.0119, 0.0690, 0.0197, 0.0078, 0.0210, 0.1484, 0.0289, 0.1416, 0.0208,
        0.0276, 0.0617, 0.0041, 0.0261, 0.0086, 0.0542, 0.0180, 0.0086, 0.0070])
label (actual next character): 5
probability assigned by the net to the the correct character: 0.06029047816991806
log likelihood: -2.8085811138153076
negative log likelihood: 2.8085811138153076
--------
bigram example 2: em (indexes 5,13)
input to the neural net: 5
output probabilities from the neural net: tensor([0.0411, 0.0255, 0.0568, 0.1050, 0.0092, 0.1304, 0.0095, 0.0241, 0.0814,
        0.0519, 0.0083, 0.0268, 0.0281, 0.0156, 0.0106, 0.0507, 0.0601, 0.0541,
        0.0180, 0.0055, 0.0432, 0.0525, 0.0122, 0.0132, 0.0175, 0.0231, 0.0255])
label (actual next character): 13
probability assigned by the net to the the correct character: 

In [79]:
# Trying to optimize the the Neural Network using gradient decent

# randomly initialize 27 neurons' weights. each neuron receives 27 inputs
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)


In [89]:
# forward pass
xenc = F.one_hot(input, num_classes=27).float() # input to the network: one-hot encoding
logits = xenc @ W # predict log-counts
counts = logits.exp() # counts, equivalent to N
probs = counts / counts.sum(1, keepdims=True) # probabilities for next character
loss = -probs[torch.arange(5), target].log().mean()

In [90]:
print(loss.item())

3.1815381050109863


In [91]:
# backward pass
W.grad = None # set to zero the gradient
loss.backward()

In [93]:
# Update
W.data += -1 * W.grad
# Changing the weights based on the gradient

In [94]:
# Optimizing on the whole dataset

In [95]:
# create the dataset
xs, ys = [], []
for w in words:
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2 in zip(chs, chs[1:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    xs.append(ix1)
    ys.append(ix2)
xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()
print('number of examples: ', num)

# initialize the 'network'
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)

number of examples:  228146


In [101]:
# gradient descent
for k in range(1):
  
  # forward pass
  xenc = F.one_hot(xs, num_classes=27).float() # input to the network: one-hot encoding
  logits = xenc @ W # predict log-counts
  counts = logits.exp() # counts, equivalent to N
  probs = counts / counts.sum(1, keepdims=True) # probabilities for next character
  loss = -probs[torch.arange(num), ys].log().mean() + 0.01*(W**2).mean()
  print(loss.item())
  
  # backward pass
  W.grad = None # set to zero the gradient
  loss.backward()
  
  # update
  W.data += -50 * W.grad

2.867231607437134
